# 第四次课后练习

<span style="color:red; font-weight:bold;">此次作业是课后练习，供大家熟练课堂上讲授的基础知识点</span>

<span style="color:red; font-weight:bold;">请将作业命名为 HW4-课后练习+姓名+学号.ipynb</span>

<span style="color:red; font-weight:bold;">提交时仅提交ipynb文件即可, 无需提交其他py文件</span>

# 第零部分 代码理解

请认真阅读代码，理解代码的功能，观察运行的结果。建议按照自己的理解调整代码，运行并检验新的结果是否如预期。如果不如预期，请分析理解其中的原因。

## **0.1** new函数用法：单例模式、不可变对象

In [1]:
class Singleton1:
    _instance = None
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls.value = 100
        return cls._instance

a = Singleton1()
a.value = 200
b = Singleton1()
print(a.value) 
print(b.value) 
print(Singleton1.value)   

200
200
100


由于每次建立实例前都要先运行__new__,故指向的是同一个实例，所以value也会一起改变，但是类的属性不会改变。

尽管 __new__ 是静态方法，但 Python 解释器在调用它时会自动传入类作为第一个参数（通常命名为 cls）。这与其他语言中的静态方法不同，可以被重载（ Python 为此设计了特殊的解析机制）以方便实现构造函数的灵活性需求。比如构造一个可以做数据预处理的构造函数：

In [ ]:
class ImmutableTuple(tuple):  # 继承tuple的所有方法-属性
    def __new__(cls, data):
        cleaned_data = [x for x in data if x is not None]   # 对输入数据进行初始化过滤
        return super().__new__(cls, cleaned_data)  # 关键：传递处理后的数据给 __new__
    
    def __init__(self, data):
        # 如果 __new__ 已经正确初始化，这里可以什么都不做
        # 或者调用 super().__init__() 但不传递参数
        pass

t = ImmutableTuple([1, None, 2, None, 3]) # 虽然是实例但是可以返回lied tuple，因为 __new__ 已经处理过了
print(t)  # [1, 2, 3]

(1, 2, 3)


In [15]:
# 实现一个类工厂
class Number:
    def __new__(cls, value):
        if isinstance(value, int):
            return super().__new__(Integer)  # 动态改变类
        elif isinstance(value, float):
            return super().__new__(Float)
        else:
            raise TypeError("不支持的类型")
    
    def __init__(self, value):
        print(f"初始化 {self.__class__.__name__} 实例")
        self.value = value

class Integer(Number):
    pass

class Float(Number):
    pass

# 测试
num1 = Number(10)   # 返回 Integer 实例
num2 = Number(3.14) # 返回 Float 实例
print(type(num1))  # <class '__main__.Integer'>
print(type(num2))  # <class '__main__.Float'>

初始化 Integer 实例
初始化 Float 实例
<class '__main__.Integer'>
<class '__main__.Float'>


继承不同的父类，实现不同的功能

## 02 闭包中的可变对象独立性

In [3]:
def make_container():
    items = []  # 闭包环境变量中的可变对象
    def add(item):
        nonlocal items
        items.append(item)
    def get():
        return items
    return {'add': add, 'get': get}  # 返回的词典包含两个函数

c1 = make_container()
c2 = make_container()  

c1['add'](1)
c1['add'](2)
c2['add'](3)

print(c1['get']())  # [1] 
print(c2['get']())  # [2] - 独立于C1？

[1, 2]
[3]


In [1]:
# 请修改上面的代码，实现多次生成（返回）的闭包函数共享可变对象item[] 
# 可以用AI生成，但要做到能理解AI返回的多种不同方案
# to do
def make_container(items=[]):  # 闭包环境变量中的可变对象
    def add(item):
        nonlocal items
        items.append(item)
    def get():
        return items
    return {'add': add, 'get': get}  # 返回的词典包含两个函数

c1 = make_container()
c2 = make_container()  

c1['add'](1)
c1['add'](2)
c2['add'](3)

print(c1['get']())  # [1] 
print(c2['get']())  # [2] - 独立于C1？




[1, 2, 3]
[1, 2, 3]


要实现多次调用 `make_container` 返回的闭包函数共享同一个可变对象 `items`，关键在于让所有闭包访问同一个列表对象，而不是每次调用都创建新的列表。以下是几种常见的实现方案，每种方案都保持了原代码返回字典（包含 `add` 和 `get` 函数）的结构。

---

### 方案一：使用函数属性（推荐）
将 `items` 作为函数 `make_container` 自身的属性，所有闭包通过函数名访问该属性，实现共享。

```python
def make_container():
    if not hasattr(make_container, "items"):
        make_container.items = []  # 初始化函数属性

    def add(item):
        make_container.items.append(item)

    def get():
        return make_container.items

    return {'add': add, 'get': get}

# 测试
c1 = make_container()
c2 = make_container()
c1['add']('apple')
c2['add']('banana')
print(c1['get']())  # 输出 ['apple', 'banana']
print(c2['get']())  # 输出 ['apple', 'banana']
```

**优点**：显式、易读，不会意外覆盖。  
**原理**：函数也是对象，可以拥有属性。`make_container.items` 在第一次调用时初始化，之后所有闭包都引用同一个列表。

---

### 方案二：利用默认参数的可变性
Python 函数的默认参数在定义时计算一次，之后每次调用如果不提供实参，则使用同一个对象。

```python
def make_container(items=[]):  # 默认列表只创建一次
    def add(item):
        items.append(item)

    def get():
        return items

    return {'add': add, 'get': get}

# 测试
c1 = make_container()
c2 = make_container()
c1['add']('apple')
c2['add']('banana')
print(c1['get']())  # 输出 ['apple', 'banana']
print(c2['get']())  # 输出 ['apple', 'banana']
```

**优点**：代码简洁。  
**缺点**：若调用时显式传入其他列表，则会打破共享（但题目未要求此场景）。默认参数为可变对象有时会引发意外，但这里正是所需。

---

### 方案三：使用全局变量
将 `items` 定义在模块级别，所有闭包直接引用全局变量。

```python
_items = []  # 全局共享列表

def make_container():
    def add(item):
        _items.append(item)

    def get():
        return _items

    return {'add': add, 'get': get}

# 测试
c1 = make_container()
c2 = make_container()
c1['add']('apple')
c2['add']('banana')
print(c1['get']())  # 输出 ['apple', 'banana']
print(c2['get']())  # 输出 ['apple', 'banana']
```

**优点**：简单直接。  
**缺点**：全局变量可能被意外修改，且污染命名空间。

---

### 方案四：使用外部嵌套作用域
创建一个外部函数，其内部定义共享列表，然后返回一个函数（生成器），每次调用该生成器获得操作同一列表的闭包。

```python
def make_container_factory():
    items = []  # 外部函数的局部变量
    def make_container():
        def add(item):
            items.append(item)

        def get():
            return items

        return {'add': add, 'get': get}
    return make_container

# 使用
factory = make_container_factory()
c1 = factory()
c2 = factory()
c1['add']('apple')
c2['add']('banana')
print(c1['get']())  # 输出 ['apple', 'banana']
print(c2['get']())  # 输出 ['apple', 'banana']
```

**优点**：数据封装在工厂函数内，不污染全局。  
**缺点**：需要多一层调用。

---

### 方案五：使用类变量
通过类来管理共享数据，实例方法操作类属性。

```python
class Container:
    items = []  # 类变量，所有实例共享

    @classmethod
    def make(cls):
        def add(item):
            cls.items.append(item)

        def get():
            return cls.items

        return {'add': add, 'get': get}

# 使用
c1 = Container.make()
c2 = Container.make()
c1['add']('apple')
c2['add']('banana')
print(c1['get']())  # 输出 ['apple', 'banana']
print(c2['get']())  # 输出 ['apple', 'banana']
```

**优点**：面向对象风格，清晰。  
**缺点**：引入了类定义。

---

### 方案六：使用单例模式（修改原代码中的 __new__ 思路）
原代码中出现了 `__new__` 的雏形，可以完善为单例类，但返回的是闭包函数。

```python
class _Container:
    items = []

    @classmethod
    def add(cls, item):
        cls.items.append(item)

    @classmethod
    def get(cls):
        return cls.items

def make_container():
    # 直接返回操作类属性的函数，无需实例化
    return {'add': _Container.add, 'get': _Container.get}
```

**优点**：利用类属性天然共享。  
**缺点**：稍微绕了一点。

---

### 总结
以上六种方案均能实现多次调用 `make_container` 返回的闭包共享同一个 `items` 列表。推荐使用**方案一（函数属性）**，因为它直观、无需额外作用域，且不易出错。根据具体场景和风格偏好，也可选择其他方案。

## 0.3 用递归函数生成器实现树结构的迭代遍历

In [4]:
import sys
from io import StringIO

class TreeNode:
    '''二叉搜索树节点的定义'''
    def __init__(self, val):
        self.val = val
        self.left = None
        self.right = None

class OperationTree:
    '''二叉搜索树操作'''
    def insert(self, root, val):
        '''二叉搜索树插入操作'''
        if root == None:
            root = TreeNode(val)
        elif val < root.val:
            root.left = self.insert(root.left, val)
        elif val > root.val:
            root.right = self.insert(root.right, val)
        return root

def inorder_traversal(root):
    '''生成器函数：递归实现中序遍历'''
    if root:
        yield from inorder_traversal(root.left)
        yield root.val
        yield from inorder_traversal(root.right)

# 使用示例
op = OperationTree()
root = None
values = [5, 3, 7, 2, 4, 6, 8]
for val in values:
    root = op.insert(root, val)

print("中序遍历结果：")
for val in inorder_traversal(root):
    print(val, end=" ")  

中序遍历结果：
2 3 4 5 6 7 8 

## 一、设计模式

本部分内容为课上所讲的几种设计模式的最简单的一些实现方式与使用场景

相较于这些设计模式如何通过代码实现，在面向对象设计和系统分析时，更重要的一点是决定在哪些情况下更适合使用哪几种设计模式。因此，在填写代码之余，同学们也可以尝试思考，课上所讲的这几种设计模式有哪些实际应用场景

### 1.1 工厂模式

工厂模式可以分为三类：简单工厂模式、工厂方法模式和抽象工厂模式（当然也有的地方将简单工厂模式看成是工厂方法模式的一种特例），同学们只需理解掌握课件上简单工厂模式的相关例子即可

**Task**：补充`SimpleFactory.create_product`方法

In [2]:
# 一个简单工厂类，这个工厂能够生产A和B两种产品
class SimpleFactory:
    @staticmethod
    def create_product(product_type):

        ###############################
        #--- Your code starts here ---#
        ###############################

        # TODO: product_type是一个字符串，表示用户要求制作的产品
        # 当product_type为"A"或"Product A"时，生产出一个ProductA对象并返回
        # 当product_type为"B"或"Product B"时，生产出一个ProductB对象并返回
        #       (当然，这里我们只实现最简单的情况，实际开发中通常会加上*args和**kwargs满足定制化要求)
        # 当遇到无理取闹的甲方，给出其它的product_type时，理直气壮地抛出异常表明这个产品做不了
        
        if product_type in ["A", "Product A"]:
            return ProductA()
        elif product_type in ["B", "Product B"]:
            return ProductB()
        else:
            raise ValueError("Product type not supported")
        
        ###############################
        #---  Your code ends here  ---#
        ###############################

# 定义工厂所能生产的产品类A和B
class ProductA:
    def description(self):
        return "Description of A"

class ProductB:
    def description(self):
        return "Description of B"

# 建立简单工厂
factory = SimpleFactory()

# 要求工厂生产产品A和产品B
product_a = factory.create_product("A")
product_b = factory.create_product("B")

# 查看产品A和产品B的相关属性
print(product_a.description())
print(product_b.description())

# 要求工厂生产不存在的产品C，运行以下语句时将会报错
product_c = factory.create_product("C")

# 注：课件上的例子相较于以上例子要稍微复杂一些，具体在于
# 1. 课件上为Cat类和Dog类定义了一个抽象父类Pet，而在这里我们直接定义了ProductA和ProductB两种产品
# 2. 课件上的StandardFactory类并不直接返回Cat/Dog类的实例，而是返回一个生产Cat/Dog类的工厂，因此：
#       通过课件上的代码获取Cat类的实例需要三步：StandardFactory -> CatFactory -> Cat，这种写法一般用在抽象层次更高的工厂类中
#       通过此处的代码获取ProductA的实例只需两步：SimpleFactory -> ProductA
# 但二者的基本思想是相同的，都是根据给定的产品名让工厂去生产出相应的产品

Description of A
Description of B


ValueError: Product type not supported

### 1.3 观察者模式/发布-订阅模式

简单的来说，观察者模式就是由多位观察者(Observer)观察一个对象(Subject)，被观察对象发生了某种变化时，所有观察他的对象得到通知并被自动更新

如学生-教学网可以视为一组观察者-被观察对象：某课程在教学网上**发布**（公告、作业等）时，选课同学（**订阅/观察**该课程的学生）可以收到**通知**（作业截止日期、公告内容等），所以观察者模式又称为发布-订阅模式

GUI-鼠标也可以视为一组观察者-被观察对象：鼠标**发布**一个行为（点击、滚轮等）时，GUI的各个组件（可以认为是**订阅/观察**了该鼠标的行为）收到**通知**（鼠标点击的位置、左键还是右键等），并自动**更新**（如某个按钮被按下后，更新界面的显示内容）

**思考**：发布-订阅模式的更多使用场景？

**Task**：在下面的代码中补充`MathCourse.add_student`和`MathCourse.remove_student`函数，模拟学生选课和退课的过程

In [ ]:
from abc import ABC, abstractmethod

# 观察者抽象类 - 学生类
class Student(ABC):
    def __init__(self, name):
        self.name = name
    @abstractmethod
    def learn_math(self):
        pass

# 观察者类 - 文科学生
class LiberalArtsStudent(Student):
    def __init__(self, name):
        super().__init__(name)
    def learn_math(self):
        print(f"{self.name}: 退了退了")

# 观察者类 - 理科学生
class ScienceStudent(Student):
    def __init__(self, name):
        super().__init__(name)
    def learn_math(self):
        print(f"{self.name}: 开卷！")

# 被观察对象类 - 数学课
class MathCourse:
    def __init__(self):
        self.students = []

    # 教学网能够增删学生
    def add_student(self, student):

        ###############################
        #--- Your code starts here ---#
        ###############################

        if isinstance(student, Student):
            self.students.append(student)
        else:
            print("Error: 学生类型错误！")
        
        ###############################
        #---  Your code ends here  ---#
        ###############################

    def remove_student(self, student):
        
        ###############################
        #--- Your code starts here ---#
        ###############################

        if student in self.students:
            self.students.remove(student)
        else:
            print("Error: 学生不存在！")
        
        ###############################
        #---  Your code ends here  ---#
        ###############################

    # 数学课通过教学网给学生发通知
    def notify_student(self):
        print("你有新的作业订单~请及时处理！")
        for student in self.students:
            student.learn_math()

# 教学网的一个实例
math_course = MathCourse()

# 文理科学生的一个实例
liberal_arts_student = LiberalArtsStudent('ART')
science_student = ScienceStudent('SCI')

# 两名学生选了课（成为观察者）
print('====选课阶段====')
math_course.add_student(liberal_arts_student)
math_course.add_student(science_student)
print(f'当前选课名单: {[stu.name for stu in math_course.students]}')
print()

# 课程发布了作业并通知选课同学
math_course.notify_student()
print()

# 文科学生退课（不再作为观察者）
print('====补退选阶段====')
math_course.remove_student(liberal_arts_student)
print(f'当前选课名单: {[stu.name for stu in math_course.students]}')
print()

# 课程发布了第二次作业并通知选课同学
math_course.notify_student()
print()

====选课阶段====
ART已选课
SCI已选课
当前选课名单: ['ART', 'SCI']

你有新的作业订单~请及时处理！
ART: 退了退了
SCI: 开卷！

====补退选阶段====
ART已退课
当前选课名单: ['SCI']

你有新的作业订单~请及时处理！
SCI: 开卷！



## 二、补充材料：类型提示与docstring简介

**本部分为补充材料，供学有余力的同学参考阅读**

本部分所介绍的内容为Python中两种特殊“注释”的写法, 希望能帮助想进一步利用Python进行开发工作的同学写出更加规范、清晰的工程代码

### 2.1 类型提示

虽然Python是动态编程语言, 不需要对变量进行类型声明, 但在Python 3.5中, 加入了“[函数参数与返回值类型提示](https://docs.python.org/3/library/typing.html)”的功能; 在Python 3.6中, 加入了“[变量声明时类型注解](https://peps.python.org/pep-0526/)”的功能.

两者的使用格式如下
```python
def func(
        var_1: type_1 [= default_val_1], 
        var_2: type_2 [= default_val_2], 
        ..., 
        var_n: type_n [= default_val_n]
    ) -> result_type:
    pass

variable: type_ = ...
```

以下是一个更具体的例子, 在这个例子中, 我们定义了一个函数`sumint`, 用于求两个整数的和, 函数参数为`a`和`b`, 其中`a:int`表示参数`a`应具有`int`类型; `b:int=3`表示参数`b`应具有`int`类型且默认值为3, 参数表后面的`-> int`表示函数的返回值是`int`类型.

我们还定义了一个变量`c`来存储`sumint(3,5)`, 在定义时, 用`c:int = ...`表示`c`应该是一个`int`类型

In [1]:
def sumint(a:int, b:int=3) -> int:
    return a + b 

c:int = sumint(3, 5)
print(c, type(c))

8 <class 'int'>


此外, Python还提供了`typing`模块, 用于更复杂的类型提示, 以下用几个例子表明这个模块的部分用法

In [2]:
from typing import Tuple, List, Dict, Set, Union

# 使用Tuple表明类型为元组
t1: Tuple = (1, 2, 3)
# 使用Tuple[type_]表明类型为元组, 元组中元素的类型为type_
t2: Tuple[str] = ('1', '2', '3')
# 使用Tuple[type1, type2, ...]表明元组各位置元素的类型
t3: Tuple[int, str, float] = (1, '2', 3.1)

# List用法与元组相似
# 但List不支持第三种用法, 运行下面这行代码会报错
# l3: List[int, str, float] = [1, "2", 3.1]

# 使用Dict表明类型为字典
d1: Dict = {1: 'a'}
# 使用Dict[type_key, type_val]表明字典键和值的类型
d2: Dict[int, str] = {2, 'b'}

# Set用法与List相似, 例子略

# 使用Union表示多种类型的“或”
# 下例表明列表内的元素类型都是int或str
l1: List[Union[int, str]] = ['2', 1]

In [3]:
from typing import Optional 

# Optional常用于函数传参, 表明某个变量可以是None
# 下例表明函数func的参数var或者是None, 或者是一个int类型的
def func(var: Optional[int] = None) -> str:
    print(var)
    return '111'

_ = func()
_ = func(3)

None
3


以上所展示的例子看起来很美好, 但事实上, 我们所加的注解和提示完全不会进行任何检查, 下面的代码仍然可以正常运行, 不会报出任何错误 (甚至也不会有Warning)

In [4]:
# 尽管我们声明a的类型为int, 但仍能将浮点数3.1赋给a
a: int = 3.1
# 当我们打印a的类型时, 输出的其实是float
print(a, type(a))

3.1 <class 'float'>


In [5]:
def sumint(a:int, b:int=2) -> int:
    return a + b

# 尽管我们声明参数a, b的类型为int, 但仍能将浮点数作为参数传入
c = sumint(1.2, 2.3)
# 尽管我们声明函数返回值的类型为int, 但打印出来发现其实是float
print(c, type(c))

3.5 <class 'float'>


虽然无法对类型进行检查和报错, 但我们仍然建议在使用Python进行开发, 特别是中型、大型项目时, 仍在必要的地方(如关键函数的参数和返回值等)加上类型提示, 以便

- 合作者/几个月后的自己阅读代码更轻松
- IDE进行代码提示
- 使用`mypy`进行代码检查, 以下简单展示mypy的用法

In [ ]:
%pip install mypy

In [7]:
%%writefile test_mypy.py 
def add(x: int, y: int) -> int:
    result = x + y
    print(result)
    return result 

add("have a ", "try!") 

Writing test_mypy.py


In [8]:
!mypy test_mypy.py

test_mypy.py:6: error: Argument 1 to "add" has incompatible type "str"; expected "int"  [arg-type]
test_mypy.py:6: error: Argument 2 to "add" has incompatible type "str"; expected "int"  [arg-type]
Found 2 errors in 1 file (checked 1 source file)


### 2.2 Docstring

Docsting可以认为是对一个函数、类、模块整体的注释或描述语句:

- 它必须紧跟在定义函数、类等语句的冒号之后, 用三引号括起
- 它会成为那个对象的`__doc__`属性
- 它的写法遵循一些规范（或者说[公约](https://peps.python.org/pep-0257/)）

In [9]:
# 例子, 为函数写单行docstring并打印它的__doc__属性

def sumint(a, b):
    '''This funcion sums two integers and output the result'''
    return a + b 

print('====  直接打印__doc__  ====')
print(sumint.__doc__)
print()

# 也可以使用help来查看__doc__
print('====  使用help查看__doc__  ====')
help(sumint)

====  直接打印__doc__  ====
This funcion sums two integers and output the result

====  使用help查看__doc__  ====
Help on function sumint in module __main__:

sumint(a, b)
    This funcion sums two integers and output the result



In [10]:
# 当描述语句并不紧跟在定义的冒号之后时, 它不会成为对象的__doc__属性

def sumint2(a, b):
    c = a + b
    '''This funcion sums two integers and output the result'''
    return c

print('====  直接打印__doc__  ====')
print(sumint2.__doc__)
print()

print('====  使用help查看__doc__  ====')
help(sumint2)

====  直接打印__doc__  ====
None

====  使用help查看__doc__  ====
Help on function sumint2 in module __main__:

sumint2(a, b)



以函数为例, 函数的docstring的内容通常可以包括函数功能、参数类型与作用、返回值类型与描述、使用时的注意事项等.

通过学习部分优秀开源代码的docstring, 可以进一步体会到docstring的特点与写法, 以下是[PyTorch](https://pytorch.org/)某类方法的[源代码](https://pytorch.org/docs/stable/_modules/torch/nn/modules/transformer.html#Transformer.forward), 大家无需看懂这个函数在做什么, 只需从中体会到写docstring的(其中一种)范式：

- 函数的功能(开头第一句话)
- 参数的作用(`Args:`下的内容)、默认值的含义(`Default:`后的内容)、注意事项(`Warning:`后的内容)
- 参数的限制(通俗的说, 这个方法实现的是矩阵的一些运算, 因此对参数形状有所要求, 这就是`Shape:`所描述的内容)和注意事项(`Note:`后的内容)
- 使用或运行的例子(`Examples:`下的内容)

```python
class Transformer:
    def forward(self, src: Tensor, tgt: Tensor, src_mask: Optional[Tensor] = None, tgt_mask: Optional[Tensor] = None,
                memory_mask: Optional[Tensor] = None, src_key_padding_mask: Optional[Tensor] = None,
                tgt_key_padding_mask: Optional[Tensor] = None, memory_key_padding_mask: Optional[Tensor] = None,
                src_is_causal: Optional[bool] = None, tgt_is_causal: Optional[bool] = None,
                memory_is_causal: bool = False) -> Tensor:
        r"""Take in and process masked source/target sequences.

        Args:
            src: the sequence to the encoder (required).
            tgt: the sequence to the decoder (required).
            src_mask: the additive mask for the src sequence (optional).
            tgt_mask: the additive mask for the tgt sequence (optional).
            memory_mask: the additive mask for the encoder output (optional).
            src_key_padding_mask: the Tensor mask for src keys per batch (optional).
            tgt_key_padding_mask: the Tensor mask for tgt keys per batch (optional).
            memory_key_padding_mask: the Tensor mask for memory keys per batch (optional).
            src_is_causal: If specified, applies a causal mask as ``src_mask``.
                Default: ``None``; try to detect a causal mask.
                Warning:
                ``src_is_causal`` provides a hint that ``src_mask`` is
                the causal mask. Providing incorrect hints can result in
                incorrect execution, including forward and backward
                compatibility.
            tgt_is_causal: If specified, applies a causal mask as ``tgt_mask``.
                Default: ``None``; try to detect a causal mask.
                Warning:
                ``tgt_is_causal`` provides a hint that ``tgt_mask`` is
                the causal mask. Providing incorrect hints can result in
                incorrect execution, including forward and backward
                compatibility.
            memory_is_causal: If specified, applies a causal mask as
                ``memory_mask``.
                Default: ``False``.
                Warning:
                ``memory_is_causal`` provides a hint that
                ``memory_mask`` is the causal mask. Providing incorrect
                hints can result in incorrect execution, including
                forward and backward compatibility.

        Shape:
            - src: :math:`(S, E)` for unbatched input, :math:`(S, N, E)` if `batch_first=False` or
              `(N, S, E)` if `batch_first=True`.
            - tgt: :math:`(T, E)` for unbatched input, :math:`(T, N, E)` if `batch_first=False` or
              `(N, T, E)` if `batch_first=True`.
            - src_mask: :math:`(S, S)` or :math:`(N\cdot\text{num\_heads}, S, S)`.
            - tgt_mask: :math:`(T, T)` or :math:`(N\cdot\text{num\_heads}, T, T)`.
            - memory_mask: :math:`(T, S)`.
            - src_key_padding_mask: :math:`(S)` for unbatched input otherwise :math:`(N, S)`.
            - tgt_key_padding_mask: :math:`(T)` for unbatched input otherwise :math:`(N, T)`.
            - memory_key_padding_mask: :math:`(S)` for unbatched input otherwise :math:`(N, S)`.

            Note: [src/tgt/memory]_mask ensures that position i is allowed to attend the unmasked
            positions. If a BoolTensor is provided, positions with ``True``
            are not allowed to attend while ``False`` values will be unchanged. If a FloatTensor
            is provided, it will be added to the attention weight.
            [src/tgt/memory]_key_padding_mask provides specified elements in the key to be ignored by
            the attention. If a BoolTensor is provided, the positions with the
            value of ``True`` will be ignored while the position with the value of ``False`` will be unchanged.

            - output: :math:`(T, E)` for unbatched input, :math:`(T, N, E)` if `batch_first=False` or
              `(N, T, E)` if `batch_first=True`.

            Note: Due to the multi-head attention architecture in the transformer model,
            the output sequence length of a transformer is same as the input sequence
            (i.e. target) length of the decoder.

            where S is the source sequence length, T is the target sequence length, N is the
            batch size, E is the feature number

        Examples:
            >>> # xdoctest: +SKIP
            >>> output = transformer_model(src, tgt, src_mask=src_mask, tgt_mask=tgt_mask)
        """
        is_batched = src.dim() == 3
        if not self.batch_first and src.size(1) != tgt.size(1) and is_batched:
            raise RuntimeError("the batch number of src and tgt must be equal")
        elif self.batch_first and src.size(0) != tgt.size(0) and is_batched:
            raise RuntimeError("the batch number of src and tgt must be equal")

        if src.size(-1) != self.d_model or tgt.size(-1) != self.d_model:
            raise RuntimeError("the feature number of src and tgt must be equal to d_model")

        memory = self.encoder(src, mask=src_mask, src_key_padding_mask=src_key_padding_mask,
                              is_causal=src_is_causal)
        output = self.decoder(tgt, memory, tgt_mask=tgt_mask, memory_mask=memory_mask,
                              tgt_key_padding_mask=tgt_key_padding_mask,
                              memory_key_padding_mask=memory_key_padding_mask,
                              tgt_is_causal=tgt_is_causal, memory_is_causal=memory_is_causal)
        return output
```

### 本次作业不单独布置选做题，有兴趣的同学可以把前一次作业中的选做题（同学选课+优化问题）改成订阅-观察模式。能完成的可以给自己的文件后加一个#

In [ ]:
# todo

